In [ ]:
"""
Document Indexing Pipeline for Hybrid Search

This notebook builds the retrieval indexes used by the HybridSearchAgent:
  - Chroma vector store  : stores OpenAI-embedded document chunks for semantic search
  - SQLite FTS5 index    : stores raw chunk text for keyword (full-text) search

Workflow
--------
1. Load documents (.txt, .pdf, .docx) from the `documents/` directory.
2. Build a Pandas DataFrame to inspect and optionally enrich document-level metadata
   (filename, folder, file type, word/char counts, custom categories, timestamps).
3. Persist the DataFrame as CSV for reproducibility.
4. Convert back to LangChain Documents and split into overlapping chunks.
5. Annotate chunks with positional metadata (chunk_id, start_char, end_char).
6. Validate and standardize chunk metadata with a Pydantic model.
7. Embed chunks and upsert into Chroma.
8. Populate the SQLite FTS5 table used by the full-text search layer.
9. PCA visualization of the resulting embedding space (optional, 2-D scatter plot).

Run the cells top-to-bottom on first use; re-run individual sections to
refresh either index after document changes.
"""

In [ ]:
# Standard library
import os
from datetime import datetime
import hashlib
from datetime import datetime, timezone

# Third-party
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, Docx2txtLoader
from langdetect import detect

# Local imports
from fts_search import FTSStore
from pydantic_models import ChunkMetadata

# Load env from specific path
load_dotenv("C:/Users/kjosi/dotenv/.env")

### LOAD DOCUMENTS AND CREATE PANDAS DATAFRAME

In [ ]:
# Load documents with DirectoryLoader - loads multiple files from a specified directory into LangChain Document objects
txt_loader = DirectoryLoader("./documents", glob="**/*.txt", loader_cls=TextLoader)
pdf_loader = DirectoryLoader("./documents", glob="**/*.pdf", loader_cls=PyPDFLoader)
docx_loader = DirectoryLoader("./documents", glob="**/*.docx", loader_cls=Docx2txtLoader)

documents = txt_loader.load() + pdf_loader.load() + docx_loader.load()
print(f"Loaded {len(documents)} documents")

In [ ]:
# Print one of the documents
print(documents[0])

In [ ]:
# Create pandas dataframe from documents and metadata
rows = []
for doc in documents:
    rows.append({
        "text": doc.page_content,
        **doc.metadata
    })

df = pd.DataFrame(rows)
df.head(2)

### OPTIONS FOR ADDING METADATA

In [ ]:
## EITHER: REQUIRED STEP - KEEP "text" and "source" 
# Drop columns - Keep "text" and "source"
df = df.drop(["list_columns_to_dorp"], axis=1)
df.head(2)

In [ ]:
## OR: REQUIRED STEP - KEEP "text" and "source"
# Columns to keep - Keep only "text" and "source" columns
df = df[["text", "source"]]
df.head(2)

In [ ]:
## REQUIRED STEP
# Add unique document IDs based on the document text itself
df['doc_hash_id'] = df['text'].apply(lambda x: hashlib.md5(x.encode()).hexdigest())
df.head(2)

In [ ]:
# Add filename from source path
df["filename"] = df["source"].apply(lambda x: os.path.basename(x))
df.head(2)

In [ ]:
# Add file type
df["file_type"] = df["filename"].apply(lambda x: os.path.splitext(x)[-1])
df.head(2)

In [ ]:
# Add folder name from source path
df["folder"] = df["source"].apply(lambda x: os.path.dirname(x))
df.head(2)

In [ ]:
# Add text length
df["doc_char_count"] = df["text"].apply(len)
df.head(2)

In [ ]:
# Add word count
df["doc_word_count"] = df["text"].apply(lambda x: len(x.split()))
df.head(2)

In [ ]:
## REQUIRED STEP
# Add ingestion timestamp
df["ingested_at"] = int(datetime.now(timezone.utc).timestamp())
df.head(2)

In [ ]:
# Add category - example: "finance" - under dev (match/switch case) - under development !!
def classify_doc(text):
    if "norway" in text.lower():
        return "Norway"
    return "other"

df["category"] = df["text"].apply(classify_doc)
df.head(2)

In [ ]:
# Add language
df['language'] = df['text'].apply(lambda x: detect(x) if x.strip() else 'unknown')
df.head(2)

In [ ]:
## OPTIONAL STEP - under development !!
# Add short summary
def generate_summary(text):
    # Call an LLM 
    # Or just take the first 100 characters as a 'mini-summary'
    return f"Snippet about: {text[:100]}..."

df['summary'] = df['text'].apply(generate_summary)
df.head(2)

In [ ]:
## OPTIONAL STEP - under development !!
# Add content type
def classify_content(text: str) -> str:
    # 1. Check for Tables (look for common markers like pipes or many tabs)
    if "|" in text or text.count("\t") > 3:
        return "table"
    # 2. Check for Headers (Short, usually no ending period, often bold/caps)
    if len(text) < 120 and not text.strip().endswith('.'):
        return "header"
    # 3. Default
    return "text"

# Apply this to your chunk-level DataFrame
df['content_type'] = df['page_content'].apply(classify_content)
df.head(2)

In [ ]:
## OPTIONAL STEP
# Save df for temporary storage as csv, save for reproducibility
df.to_csv("documents_with_metadata_df.csv", index=False)

# Load file later
df = pd.read_csv("documents_with_metadata_df.csv")

### CONVERT BACK FROM PANDAS DATAFRAME TO DOCUMENTS

In [ ]:
# Convert back to LangChain Document objects with metadata
documents_with_metadata = []

for _, row in df.iterrows(): 
    metadata = row.drop("text").to_dict()

    documents_with_metadata.append(
        Document(
            page_content=row["text"],
            metadata=metadata
        )
    )

print(f"Loaded {len(documents)} documents")

In [ ]:
# Print one of the documents
print(documents_with_metadata[0])

In [ ]:
## OPTIONAL STEP
# If small tweaks are needed, change metadata after document creation, example: category
for doc in documents_with_metadata:
    doc.metadata["category"] = "new_value"

### CREATE CHUNKS AND ADD CHUNK METADATA

In [ ]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents_with_metadata)

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(chunks):
    print(chunk.metadata)

In [ ]:
# Print specific chunk metadata, example: source
for i, chunk in enumerate(chunks):
    print(chunk.metadata["source"])

In [ ]:
## REQUIRED STEP
# Add chunk_id
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

In [ ]:
## REQUIRED STEP
# Add chunk_hash_id - unique ID for each chunk based on doc_id + chunk_id
for i, chunk in enumerate(chunks):
    chunk_id_string = f"{chunk.metadata['doc_hash_id']}_{i}"
    chunk.metadata["chunk_hash_id"] = hashlib.md5(chunk_id_string.encode()).hexdigest()

In [ ]:
# Add chunk start_char and end_char
start = 0

for i, chunk in enumerate(chunks):
    text = chunk.page_content
    end = start + len(text)

    chunk.metadata["chunk_start_char"] = start
    chunk.metadata["chunk_end_char"] = end

    start = end

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(chunks):
    print(chunk.metadata)

In [ ]:
# Print specific chunk metadata, example: chunk_hash_id
for i, chunk in enumerate(chunks):
    print(chunk.metadata["chunk_hash_id"])

### STANDARDIZE CHUNK METADATA

In [ ]:
# Definition to standardize chunk metadata
def standardize_chunk_metadata(chunk):
    """
    Standardizes chunk metadata using Pydantic for type validation.

    Parses metadata into structured fields (e.g., "3" -> 3) based on 
    ChunkMetadata and isolates any remaining extra fields.

    Returns:
        tuple: (original chunk, structured_dict, extra_fields_dict)
    """

    # Explain, example: "chunk_id": "3" > Pydantic converts it > 3 (int)
    std_meta = ChunkMetadata.model_validate(chunk.metadata)

    # model_dump() converts pydandic model into a standard python dict
    structured_metadata = std_meta.model_dump() 

    # Extracts "extra metadata", everything that is NOT in the schema
    extra_metadata = {key: value for key, value in chunk.metadata.items() if key not in structured_metadata}

    return chunk, structured_metadata, extra_metadata

In [ ]:
# Run standardize_chunk_metadata() function on all chunks
clean_chunks = []

for chunk in chunks:
    _, structured_metadata, extra_metada = standardize_chunk_metadata(chunk)

    clean_chunk = chunk.model_copy()
    clean_chunk.metadata = {**structured_metadata, **extra_metada}

    clean_chunks.append(clean_chunk)

### INGEST CHUNKS: CHROMA DB AND SQLite FTS

In [ ]:
# Create / load Chroma DB (persisted locally) - when DB does not already exist
vector_store = Chroma(
    collection_name="document_collection_1",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)

# Add chunks to Chroma db
vector_store.add_documents(clean_chunks)

print("Embeddings stored in ./chroma_db")

In [ ]:
## OPTIONAL STEP — Refresh and upsert Chroma DB
# Use this cell to synchronize your Vector Store with the new chunks.
# This approach ensures version consistency and prevents "ghost chunks" from old files.

# Set to True to wipe all existing chunks for a 'source' before inserting the new version.
# Set to False for a pure upsert (only updates chunks with identical chunk_hash_id).
REFRESH_EXISTING_SOURCES = True

# Initialize the Chroma vector store
vector_store = Chroma(
    collection_name="document_collection_1",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)

# Prepare batch data for the database
# Use pre-calculated chunk_hash_id as the primary database ID for deterministic lookups.
ids = [chunk.metadata.get("chunk_hash_id") for chunk in clean_chunks]
documents = [chunk.page_content for chunk in clean_chunks]
metadatas = [chunk.metadata for chunk in clean_chunks]

# Generate embeddings for the current batch
embeddings = OpenAIEmbeddings().embed_documents(documents)

# --- SOURCE-LEVEL CLEANUP ---
# If enabled, we remove all previous chunks belonging to the files in this batch.
# This is critical if the new file version has fewer chunks than the old version.
if REFRESH_EXISTING_SOURCES:
    
    # Identify unique filenames (sources) in the current batch to refresh.
    # The Set {} removes duplicates so we only perform one delete per document.
    sources_to_refresh = sorted({m.get("source") for m in metadatas if m.get("source")})
    
    # Iterate through each unique source and purge old data.
    # This matches the 'source' metadata field stored in Chroma.
    for source in sources_to_refresh:
        # Deletes all chunks associated with this file source
        vector_store._collection.delete(where={"source": source})
    
    print(f"Chroma DB: Purged existing chunks for {len(sources_to_refresh)} source(s)")


# --- UPSERT DATA ---
# We use _collection.upsert to either insert new chunks or update existing ones.
# Because we provide specific IDs (chunk_hash_id), we avoid creating duplicates.
vector_store._collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print(f"Chroma DB: Successfully upserted {len(clean_chunks)} chunks.")

In [ ]:
# Initiate FTS and index chunks - when FTS index does not already exist
fts_store = FTSStore()
fts_store.add_documents(clean_chunks)

print("Indexing complete")

In [ ]:
## OPTIONAL STEP — UPDATE SQLite FTS (delete + re-index)
# Use this cell to synchronize the SQLite Full-Text Search (FTS) index with the new chunks.
# It ensures that only the latest version of each document exists in the database.

# Set to True to perform a "Wipe and Replace" for every file in the current batch.
# Set to False to skip existing files and only add brand-new ones.
REFRESH_EXISTING_SOURCES = True

# Initialize or connect to the FTS storage
fts_store = FTSStore()

# Get sorted set with unique source values from the new chunks 
sources_to_refresh = sorted({m.get("source") for m in [c.metadata for c in clean_chunks] if m.get("source")})

if REFRESH_EXISTING_SOURCES:
    """
    Perform a Source-Level Refresh:
    1. Iterates through each unique source in the new batch.
    2. Deletes all existing rows in SQLite matching that source.
       - This prevents 'ghost' chunks if the new file version is shorter than the old one.
    3. Re-inserts the fresh chunks with updated content and metadata (like doc_hash_id).
    """

    for source in sources_to_refresh:
        fts_store.cur.execute("DELETE FROM docs WHERE source = ?", (source,))
    
    fts_store.conn.commit()
    print(f"SQLite FTS: Purged existing rows for {len(sources_to_refresh)} source(s)")

    fts_store.add_documents(clean_chunks)
    print(f"SQLite FTS: Re-indexed {len(clean_chunks)} chunks.")

else:
    """
    Perform an Append-Only Update:
    1. Fetches a list of all sources already present in the database.
    2. Filters the current batch to find chunks from completely new files.
    3. Only inserts chunks that don't have a matching source in the DB.
    """

    fts_store.cur.execute("SELECT DISTINCT source FROM docs")
    indexed_sources = {row[0] for row in fts_store.cur.fetchall()}
    
    new_chunks = [c for c in clean_chunks if c.metadata.get("source") not in indexed_sources]
    if new_chunks:
        fts_store.add_documents(new_chunks)
        print(f"SQLite FTS: Added {len(new_chunks)} chunks from new sources.")
    else:
        print("SQLite FTS: All sources already indexed. No new chunks added.")


### CHECK DOCUMENTS, CHUNKS, AND METADATA

In [ ]:
# View what is stored in persisted vector store
vector_store = Chroma(
    collection_name="document_collection_1",
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)

print(vector_store.get())

In [ ]:
# Print chunk metadata
for i, chunk in enumerate(clean_chunks):
    print(chunk.metadata)

In [ ]:
# Print and inspect metadata
collection = vector_store._collection
data = collection.get(include=["embeddings", "documents", "metadatas"])

print(f"Number of chunks: {len(data['documents'])}\n")

for i in range(len(data["documents"])):
    print("ID:", data["ids"][i])
    print("TEXT:", data["documents"][i])
    print("METADATA:", data["metadatas"][i])
    print("VECTOR (first 5 dims):", data["embeddings"][i][:5])
    print("-"*40)

In [ ]:
# Test retrieval, metatdata
results = vector_store.similarity_search("your query", k=1)

for result in results:
    print(result.page_content)
    print(result.metadata)

In [ ]:
# Run PCA to see how documents cluster - under development !!
data = collection.get(include=["embeddings"])
embeddings = np.array(data["embeddings"])

pca = PCA(n_components=2)
reduced = pca.fit_transform(embeddings)